# corrector_01 - Build the Vietnamese sentence corpus (role B)

Downloads Vietnamese Wikipedia (`wikimedia/wikipedia`, config `20231101.vi`), cleans it with
`src/corrector/corpus.py` and writes one sentence per line to Drive:

```
data/corpus/train.txt   (about 98 %)
data/corpus/val.txt     (about 1 %)
data/corpus/test.txt    (about 1 %)
data/corpus/stats.json
```

No GPU needed: choose **Runtime > Change runtime type > CPU**. Needs the `GITHUB_TOKEN` secret with notebook access on.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN")
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

## Settings

`MAX_SENTENCES` limits the corpus size. 1.5 million sentences is plenty for a frequency table,
a bigram model and a small transformer. Building it takes roughly 15-30 minutes.

In [ ]:
MAX_SENTENCES = 1_500_000
CORPUS_DIR = f"{DATA_DIR}/corpus"
LOCAL_DIR = "/content/corpus"  # write locally first (fast), copy to Drive at the end

In [ ]:
import json, os, shutil, time
from datasets import load_dataset
from src.corrector.corpus import sentences_from_article, assign_split

os.makedirs(LOCAL_DIR, exist_ok=True)
files = {name: open(f"{LOCAL_DIR}/{name}.txt", "w", encoding="utf-8") for name in ["train", "val", "test"]}
counts = {"train": 0, "val": 0, "test": 0}
seen = set()            # hashes of sentences already written (removes duplicates)
articles = 0
duplicates = 0
start = time.time()

wiki = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
for article in wiki:
    articles += 1
    for sentence in sentences_from_article(article["text"]):
        key = hash(sentence)
        if key in seen:
            duplicates += 1
            continue
        seen.add(key)
        split = assign_split(sentence)
        files[split].write(sentence + "\n")
        counts[split] += 1
    total = sum(counts.values())
    if articles % 20_000 == 0:
        print(f"{articles:>8} articles  {total:>9} sentences  {time.time() - start:6.0f} s")
    if total >= MAX_SENTENCES:
        break

for f in files.values():
    f.close()
print("done:", articles, "articles,", counts, "duplicates skipped:", duplicates)

In [ ]:
syllables = 0
with open(f"{LOCAL_DIR}/train.txt", encoding="utf-8") as f:
    for line in f:
        syllables += len(line.split())

stats = {
    "source": "wikimedia/wikipedia 20231101.vi",
    "articles_read": articles,
    "sentences": counts,
    "duplicates_skipped": duplicates,
    "train_tokens": syllables,
}
with open(f"{LOCAL_DIR}/stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

os.makedirs(CORPUS_DIR, exist_ok=True)
for name in ["train.txt", "val.txt", "test.txt", "stats.json"]:
    shutil.copy(f"{LOCAL_DIR}/{name}", f"{CORPUS_DIR}/{name}")
print(json.dumps(stats, ensure_ascii=False, indent=2))
print("Saved to", CORPUS_DIR)

## Look at a few sentences and their noisy versions

In [ ]:
import itertools
from src.corrector.noise import make_pairs

with open(f"{CORPUS_DIR}/val.txt", encoding="utf-8") as f:
    sample = [line.strip() for line in itertools.islice(f, 10)]

for noisy, clean in make_pairs(sample, seed=0):
    print("clean:", clean)
    print("noisy:", noisy)
    print()